In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("kay_value_list","")

dbutils.widgets.text("cdc_col","")

dbutils.widgets.text("source_object","")

dbutils.widgets.text("source_schema","")

dbutils.widgets.text("target_object","")

dbutils.widgets.text("target_schema","")

In [0]:
key_value_col="['passenger_id']"
key_value_list=eval(key_value_col)


backdate_refresh=""


source_object="silver_passenger"

source_schema="silver"

target_object="dim_passenger"

target_schema="gold"

cdc_col="updated_date"

surrogate_key="dimpassengerkey"



In [0]:
if len(backdate_refresh)==0:

    if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):

        last_load=spark.sql(f"SELECT MAX({cdc_col}) FROM workspace.{target_schema}.{target_object}").collect()[0][0]
    else:

        last_load="1900-01-01 00:00:00"
else:
    last_load=backdate_refresh

last_load

datetime.datetime(2026, 5, 8, 6, 12, 59, 664000)

In [0]:
df_src=spark.sql(f"SELECT * FROM workspace.{source_schema}.{source_object} WHERE {cdc_col}>='{last_load}'")
display(df_src)


passenger_id,name,gender,nationality,updated_date
P0001,Kevin Ferguson,Male,Reunion,2026-05-08T06:12:59.664Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-05-08T06:12:59.664Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-05-08T06:12:59.664Z
P0004,Ryan Ramsey,Male,Niger,2026-05-08T06:12:59.664Z
P0005,Mike Kim,Male,Taiwan,2026-05-08T06:12:59.664Z
P0006,Diana Adams,Male,Mayotte,2026-05-08T06:12:59.664Z
P0007,Sharon Moon,Male,Madagascar,2026-05-08T06:12:59.664Z
P0008,Cheryl Glenn,Male,Maldives,2026-05-08T06:12:59.664Z
P0009,Allen Lowery,Male,Rwanda,2026-05-08T06:12:59.664Z
P0010,Maria Medina,Male,Denmark,2026-05-08T06:12:59.664Z


In [0]:
key_value_str=', '.join(key_value_list)
key_value_str

'passenger_id'

In [0]:
if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
    #it is for incremental load 
    # key value
    key_value_str=', '.join(key_value_list)
    df_trg=spark.sql(f"SELECT {key_value_str},{surrogate_key}, create_date, update_date FROM workspace.{target_schema}.{target_object}")
else:
    # it is for initial load of data
    key_value_init=[f"'' as {i}"for i in key_value_list]
    key_value_init=', '.join(key_value_init)
    df_trg=spark.sql(f"""SELECT {key_value_init},cast('0' as int) as {surrogate_key},cast('1900-01-01 00:00:00' as timestamp) as create_date, cast('1900-01-01 00:00:00'as timestamp) as update_date where 1=0""")


In [0]:

df_trg.display()

passenger_id,dimpassengerkey,create_date,update_date
P0001,1,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0002,2,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0003,3,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0004,4,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0005,5,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0006,6,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0007,7,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0008,8,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0009,9,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0010,10,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z


In [0]:
key_value_str=', '.join(key_value_list)
key_value_str


'passenger_id'

In [0]:
join_condition=' AND '.join([f"src.{i}=trg.{i}" for i in key_value_list])
join_condition

'src.passenger_id=trg.passenger_id'

In [0]:
df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")
df_join=spark.sql(f"""
        SELECT src.*,
        trg.{surrogate_key},
        trg.create_date,
        trg.update_date
        FROM src
        LEFT JOIN trg
        ON {join_condition}         
         """)
display(df_join)


passenger_id,name,gender,nationality,updated_date,dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-05-08T06:12:59.664Z,1,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-05-08T06:12:59.664Z,2,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-05-08T06:12:59.664Z,3,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0004,Ryan Ramsey,Male,Niger,2026-05-08T06:12:59.664Z,4,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0005,Mike Kim,Male,Taiwan,2026-05-08T06:12:59.664Z,5,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0006,Diana Adams,Male,Mayotte,2026-05-08T06:12:59.664Z,6,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0007,Sharon Moon,Male,Madagascar,2026-05-08T06:12:59.664Z,7,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0008,Cheryl Glenn,Male,Maldives,2026-05-08T06:12:59.664Z,8,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0009,Allen Lowery,Male,Rwanda,2026-05-08T06:12:59.664Z,9,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0010,Maria Medina,Male,Denmark,2026-05-08T06:12:59.664Z,10,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z


In [0]:
df_old=df_join.filter(df_join[surrogate_key].isNotNull())
df_new=df_join.filter(df_join[surrogate_key].isNull())

In [0]:
display(df_old)
display(df_new)

passenger_id,name,gender,nationality,updated_date,dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-05-08T06:12:59.664Z,1,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-05-08T06:12:59.664Z,2,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-05-08T06:12:59.664Z,3,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0004,Ryan Ramsey,Male,Niger,2026-05-08T06:12:59.664Z,4,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0005,Mike Kim,Male,Taiwan,2026-05-08T06:12:59.664Z,5,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0006,Diana Adams,Male,Mayotte,2026-05-08T06:12:59.664Z,6,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0007,Sharon Moon,Male,Madagascar,2026-05-08T06:12:59.664Z,7,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0008,Cheryl Glenn,Male,Maldives,2026-05-08T06:12:59.664Z,8,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0009,Allen Lowery,Male,Rwanda,2026-05-08T06:12:59.664Z,9,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z
P0010,Maria Medina,Male,Denmark,2026-05-08T06:12:59.664Z,10,2026-05-01T17:20:31.512Z,2026-05-08T10:06:39.393Z


passenger_id,name,gender,nationality,updated_date,dimpassengerkey,create_date,update_date


In [0]:
df_old_enrich=df_old.withColumn('update_date', current_timestamp())

In [0]:
if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
    max_surrogate_key=spark.sql(f"""
                            SELECT MAX({surrogate_key}) FROM workspace.{target_schema}.{target_object}
                        """).collect()[0][0]
    df_new=df_new.withColumn(f'{surrogate_key}',lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
        .withColumn('create_date',current_timestamp())\
        .withColumn('update_date',current_timestamp())
else:
    max_surrogate_key=0
    df_new=df_new.withColumn(f'{surrogate_key}',lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
        .withColumn('create_date',current_timestamp())\
        .withColumn('update_date',current_timestamp())

In [0]:
df_union=df_old_enrich.union(df_new)
df_union.display()

passenger_id,name,gender,nationality,updated_date,dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-05-08T06:12:59.664Z,1,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-05-08T06:12:59.664Z,2,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-05-08T06:12:59.664Z,3,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0004,Ryan Ramsey,Male,Niger,2026-05-08T06:12:59.664Z,4,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0005,Mike Kim,Male,Taiwan,2026-05-08T06:12:59.664Z,5,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0006,Diana Adams,Male,Mayotte,2026-05-08T06:12:59.664Z,6,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0007,Sharon Moon,Male,Madagascar,2026-05-08T06:12:59.664Z,7,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0008,Cheryl Glenn,Male,Maldives,2026-05-08T06:12:59.664Z,8,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0009,Allen Lowery,Male,Rwanda,2026-05-08T06:12:59.664Z,9,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z
P0010,Maria Medina,Male,Denmark,2026-05-08T06:12:59.664Z,10,2026-05-01T17:20:31.512Z,2026-05-08T13:52:08.865Z


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
    df_obj=DeltaTable.forName(spark,f"workspace.{target_schema}.{target_object}")
    df_obj.alias("trg").merge(df_union.alias("src"),f"src.{surrogate_key}=trg.{surrogate_key}")\
        .whenMatchedUpdateAll(condition=f"src.{cdc_col}>=trg.{cdc_col}")\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_union.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"workspace.{target_schema}.{target_object}")

In [0]:


%sql
SELECT * FROM workspace.gold.dim_passenger where passenger_id='P0049'

passenger_id,name,gender,nationality,updated_date,dimpassengerkey,create_date,update_date
P0049,Justin Thomas,Female,Tokelau,2026-05-08T06:12:59.664Z,49,2026-05-01T17:20:31.512Z,2026-05-08T13:52:11.870Z


In [0]:
%sql
SELECT * FROM workspace.gold.dim_flight ;

flight_id,airline,origin,destination,flight_date,updated_date,dimflightkey,create_date,update_date
F0001,Delta,Kellyfort,South Kathleen,2025-05-04,2026-04-25T17:10:18.261Z,1,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0002,Qatar Airways,Lake Stephen,New Vincent,2025-04-29,2026-04-25T17:10:18.261Z,2,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0003,Lufthansa,East Patrickborough,North Mary,2025-05-11,2026-04-25T17:10:18.261Z,3,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0004,Delta,Maddenshire,Johnchester,2025-05-16,2026-04-25T17:10:18.261Z,4,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0005,Qatar Airways,Bennettside,New Mistyhaven,2025-06-13,2026-04-25T17:10:18.261Z,5,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0006,Air Canada,New Richardside,South Jamesborough,2025-05-16,2026-04-25T17:10:18.261Z,6,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0007,Delta,Berryport,Miguelburgh,2025-05-24,2026-04-25T17:10:18.261Z,7,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0008,Lufthansa,Briannachester,Cervantesland,2025-05-26,2026-04-25T17:10:18.261Z,8,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0009,Delta,Alexandraborough,North Alexishaven,2025-06-10,2026-04-25T17:10:18.261Z,9,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
F0010,Emirates,Kruegerchester,Martintown,2025-05-20,2026-04-25T17:10:18.261Z,10,2026-05-01T12:53:59.849Z,2026-05-01T16:50:49.786Z
